### Formula 1 Object detection model (YOLOv26)

#### Environment

`(1) Packages` 

In [ ]:
%pip install ultralytics
%pip install fiftyone

In [ ]:
from ultralytics import YOLO
import cv2
import pandas as pd
from pathlib import Path
from collections import defaultdict
import matplotlib.pyplot as plt
import subprocess

#### Part 1: Collect Data

In [ ]:
video_folder = Path("D:/new_pc/engineershit/github_projects/Computer_Vision/datasets/videos_input")
output_folder = Path("D:/new_pc/engineershit/github_projects/Computer_Vision/datasets/output_output")

for video in video_folder.glob("*.mp4"):


    race_output = output_folder / video.stem
    race_output.mkdir(parents=True, exist_ok=True)

    output_pattern = race_output / "frame_%05d.jpg"

    subprocess.run([
        "ffmpeg",
        "-i", str(video),
        "-vf", "fps=1",
        str(output_pattern)
    ])

#### Part 2: Prepare model environment

In [ ]:
#Data
F1_CV_project_folder = Path("D:/new_pc/engineershit/github_projects/Computer_Vision/F1_CV_project")

#Save
run_name_1 = "yolo26n_f1_car_detection_50epochs"
run_name_2 = "yolo26n_f1_car_detection_100epochs"

#### Part 3: Create model

`(1) 50 epochs` 

In [ ]:
#50 epochs

model = YOLO("yolo26n.pt")

training_model = model.train(
    data="/content/drive/MyDrive/MSc_Thesis_2026/datasets/approved_dataset/data.yaml",  # data    
    epochs=50,  
    imgsz=640,   
    project=F1_CV_project_folder,
    name=run_name_1,
    exist_ok=False
)

`(2) 100 epochs` 

In [ ]:


model_100 = YOLO("yolo26n.pt")

training_model = model_100.train(
    data="/content/drive/MyDrive/MSc_Thesis_2026/datasets/approved_dataset/data.yaml",  # data    
    epochs=100,  
    imgsz=640,   
    project=F1_CV_project_folder,
    name=run_name_2,
    exist_ok=False
)

`Save best models` 

In [ ]:
best_model_50_path = "/content/drive/MyDrive/github/yolo_f1_runs/yolo26n_f1_car_detection_50epochs/weights/best.pt"
best_model_100_path = "/content/drive/MyDrive/github/yolo_f1_runs/yolo26n_f1_car_detection_100epochs/weights/best.pt"

`Evaluate` 

In [ ]:
run_50 = "/content/drive/MyDrive/github/yolo_f1_runs/yolo26n_f1_car_detection_50epochs/results.csv"
run_100 = "/content/drive/MyDrive/github/yolo_f1_runs/yolo26n_f1_car_detection_100epochs-2/results.csv"

df_50 = pd.read_csv(run_50)
df_100 = pd.read_csv(run_100)


# Remove possible spaces in column names
df_50.columns = df_50.columns.str.strip()
df_100.columns = df_100.columns.str.strip()

cols = [
    "epoch",
    "metrics/precision(B)",
    "metrics/recall(B)",
    "metrics/mAP50(B)",
    "metrics/mAP50-95(B)"
]

comparison = pd.DataFrame({
    "Model": ["YOLO26n", "YOLO26n"],
    "Epochs": [50, 100],
    "Precision": [
        df_50["metrics/precision(B)"].iloc[-1],
        df_100["metrics/precision(B)"].iloc[-1]
    ],
    "Recall": [
        df_50["metrics/recall(B)"].iloc[-1],
        df_100["metrics/recall(B)"].iloc[-1]
    ],
    "mAP50": [
        df_50["metrics/mAP50(B)"].iloc[-1],
        df_100["metrics/mAP50(B)"].iloc[-1]
    ],
    "mAP50-95": [
        df_50["metrics/mAP50-95(B)"].iloc[-1],
        df_100["metrics/mAP50-95(B)"].iloc[-1]
    ]
})

comparison

#### Part 4: Tracking and trajectory

`Define paths` 

In [ ]:
# This is the video you want to analyze.
video_path = "/content/drive/MyDrive/MSc_Thesis_2026/testing_environment/test1/input/birds_eye_barca.mp4"

# This CSV will store the extracted trajectory data.
output_csv = "/content/drive/MyDrive/github/trajectory_data/trajectory_summary.csv"

# This will be the final video with trajectory lines drawn on it.
output_video_path = "/content/drive/MyDrive/github/trajectory_data/barca_bird.mp4"

`Storage` 

In [ ]:
rows = []

`Collect coordinates` 

In [ ]:
cap = cv2.VideoCapture(video_path)


frame_idx = 0


while cap.isOpened():


    success, frame = cap.read()


    if not success:
        break


    results = model_100.track(
        frame,
        persist=True,
        tracker="bytetrack.yaml",
        conf=0.25,
        iou=0.5,
        verbose=False
    )


    result = results[0]


    if result.boxes is not None and result.boxes.id is not None:


        boxes = result.boxes


        xyxy = boxes.xyxy.cpu().numpy()


        track_ids = boxes.id.cpu().numpy()


        confidences = boxes.conf.cpu().numpy()


        class_ids = boxes.cls.cpu().numpy()


        for box, track_id, confidence, class_id in zip(
            xyxy, track_ids, confidences, class_ids
        ):


            x1, y1, x2, y2 = box


            cx = (x1 + x2) / 2
            cy = (y1 + y2) / 2


            width = x2 - x1
            height = y2 - y1


            time_seconds = frame_idx / fps


            rows.append({
                "video_path": video_path,
                "frame": frame_idx,
                "time_seconds": time_seconds,
                "track_id": int(track_id),
                "class_id": int(class_id),
                "confidence": float(confidence),
                "x1": float(x1),
                "y1": float(y1),
                "x2": float(x2),
                "y2": float(y2),
                "cx": float(cx),
                "cy": float(cy),
                "width": float(width),
                "height": float(height),
                "cx_norm": float(cx / frame_width),
                "cy_norm": float(cy / frame_height)
            })


    frame_idx += 1


    if frame_idx % 100 == 0:
        print(f"Processed frame {frame_idx}")


cap.release()

print("Tracking finished.")
print("Number of collected detections:", len(rows))

`Save data` 

In [ ]:
df = pd.DataFrame(rows)


if df.empty:
    print("No tracking data was collected.")
    print("Try checking the model, video path, or lowering conf from 0.25 to 0.15.")
else:

    Path(output_csv).parent.mkdir(parents=True, exist_ok=True)


    df.to_csv(output_csv, index=False)

    print("Saved trajectory CSV to:", output_csv)
    print("Number of rows:", len(df))
    print(df.head())

`Adding Trail - (1) Open video` 

In [ ]:
cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    raise ValueError("Video could not be opened. Check video_path.")

fps = cap.get(cv2.CAP_PROP_FPS)
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print("Video opened successfully.")
print("FPS:", fps)
print("Frame width:", frame_width)
print("Frame height:", frame_height)
print("Total frames:", total_frames)


`Adding Trail - (2) Create output video` 

In [ ]:
Path(output_video_path).parent.mkdir(parents=True, exist_ok=True)

fourcc = cv2.VideoWriter_fourcc(*"mp4v")

out = cv2.VideoWriter(
    output_video_path,
    fourcc,
    fps,
    (frame_width, frame_height)
)

`Adding Trail - (3) Trajectory settings` 

In [ ]:
# If already known which track ID you want, write it here.
# Example:
# selected_track_id = 56
#
# If unknow the ID yet, leave it as None.
# The script will automatically select the largest detected car first.
selected_track_id = None

# Choose which point of the bounding box represents the car position.
#
# "center" = middle of bounding box
# "bottom_center" = bottom-middle of bounding box, closer to road contact point
point_mode = "center"

# IMPORTANT:
# Keep only the last 5 points, so the line stays behind the car
# and does not remain permanently on the screen.
max_trail_points = 10

# If the tracker loses the selected ID briefly, this allows the script
# to follow the nearest detection instead of immediately stopping.
use_nearest_fallback = True

# Maximum pixel distance allowed for nearest fallback.
# If the nearest object is farther than this, it will not be used.
max_fallback_distance = 120

# Store recent trajectory points here.
trajectory_points = []

# Store all selected trajectory rows here for CSV output.
trajectory_rows = []

# Frame counter.
frame_idx = 0

`Adding Trail - (4) Process video and draw trail` 

In [ ]:
import math

while cap.isOpened():

    # Read one frame from the video.
    success, frame = cap.read()

    # Stop if video ended.
    if not success:
        break

    # Run YOLO detection + tracking on the current frame.
    results = model_for_100_epochs.track(
        frame,
        persist=True,
        conf=0.25,
        iou=0.5,
        verbose=False
    )

    result = results[0]

    target_point = None
    target_box = None
    target_confidence = None
    current_track_id = None

    # Continue only if detections and track IDs exist.
    if result.boxes is not None and result.boxes.id is not None:

        boxes = result.boxes

        xyxy = boxes.xyxy.cpu().numpy()
        track_ids = boxes.id.cpu().numpy()
        confidences = boxes.conf.cpu().numpy()

        detections = []

        # Collect all detections in the current frame.
        for box, track_id, confidence in zip(xyxy, track_ids, confidences):

            x1, y1, x2, y2 = box

            # Calculate representative point of the detected car.
            if point_mode == "center":
                px = (x1 + x2) / 2
                py = (y1 + y2) / 2

            elif point_mode == "bottom_center":
                px = (x1 + x2) / 2
                py = y2

            else:
                raise ValueError("point_mode must be 'center' or 'bottom_center'.")

            width = x2 - x1
            height = y2 - y1
            area = width * height

            detections.append({
                "track_id": int(track_id),
                "confidence": float(confidence),
                "box": (int(x1), int(y1), int(x2), int(y2)),
                "point": (int(px), int(py)),
                "area": float(area)
            })

        # ----------------------------------------------------
        # CASE 1: No selected ID yet.
        # Automatically choose the largest detected car.
        # ----------------------------------------------------
        if selected_track_id is None and len(detections) > 0:

            largest_detection = max(detections, key=lambda d: d["area"])

            selected_track_id = largest_detection["track_id"]

            print("Automatically selected track ID:", selected_track_id)

        # ----------------------------------------------------
        # CASE 2: Try to find the selected ID in this frame.
        # ----------------------------------------------------
        selected_detection = None

        for detection in detections:
            if detection["track_id"] == selected_track_id:
                selected_detection = detection
                break

        # ----------------------------------------------------
        # CASE 3: If selected ID is missing, use nearest fallback.
        # This helps when the tracker changes ID briefly.
        # ----------------------------------------------------
        if selected_detection is None and use_nearest_fallback and len(trajectory_points) > 0:

            last_point = trajectory_points[-1]

            nearest_detection = None
            nearest_distance = float("inf")

            for detection in detections:

                px, py = detection["point"]

                distance = math.sqrt(
                    (px - last_point[0]) ** 2 +
                    (py - last_point[1]) ** 2
                )

                if distance < nearest_distance:
                    nearest_distance = distance
                    nearest_detection = detection

            if nearest_detection is not None and nearest_distance <= max_fallback_distance:
                selected_detection = nearest_detection
                selected_track_id = selected_detection["track_id"]

        # ----------------------------------------------------
        # If selected detection exists, save its point.
        # ----------------------------------------------------
        if selected_detection is not None:

            target_point = selected_detection["point"]
            target_box = selected_detection["box"]
            target_confidence = selected_detection["confidence"]
            current_track_id = selected_detection["track_id"]

            # Add current point to the trajectory.
            trajectory_points.append(target_point)

            # Keep only the most recent 5 points.
            # This creates a short trail instead of a permanent full path.
            trajectory_points = trajectory_points[-max_trail_points:]

            # Save data for CSV output.
            trajectory_rows.append({
                "frame": frame_idx,
                "time_seconds": frame_idx / fps,
                "track_id": current_track_id,
                "x": target_point[0],
                "y": target_point[1],
                "confidence": target_confidence
            })

            # Draw bounding box around the selected car.
            x1, y1, x2, y2 = target_box

            cv2.rectangle(
                frame,
                (x1, y1),
                (x2, y2),
                (255, 0, 255),  # purple box
                2
            )

    # --------------------------------------------------------
    # Draw short purple trajectory trail.
    # --------------------------------------------------------
    if len(trajectory_points) > 1:

        for i in range(1, len(trajectory_points)):

            cv2.line(
                frame,
                trajectory_points[i - 1],
                trajectory_points[i],
                (255, 0, 255),  # purple/magenta in OpenCV
                4
            )

    # --------------------------------------------------------
    # Draw blue dots on the recent trajectory points.
    # --------------------------------------------------------
    for point in trajectory_points:

        cv2.circle(
            frame,
            point,
            3,
            (255, 0, 0),  # blue
            -1
        )

    # --------------------------------------------------------
    # Draw red dot at the latest/current position.
    # --------------------------------------------------------
    if len(trajectory_points) > 0:

        latest_point = trajectory_points[-1]

        cv2.circle(
            frame,
            latest_point,
            6,
            (0, 0, 255),  # red
            -1
        )

        cv2.putText(
            frame,
            f"ID {selected_track_id}",
            (latest_point[0] + 10, latest_point[1] - 10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (255, 255, 255),
            2
        )

    # Save edited frame.
    out.write(frame)

    # Move to next frame.
    frame_idx += 1

    if frame_idx % 100 == 0:
        print(f"Processed frame {frame_idx}")

`Adding Trail - (5) Release video` 

In [ ]:
cap.release()
out.release()

`Adding Trail - (6) Save trajectory as .csv` 

In [ ]:
trajectory_df = pd.DataFrame(trajectory_rows)

Path(output_csv).parent.mkdir(parents=True, exist_ok=True)

trajectory_df.to_csv(output_csv, index=False)

print("Saved trajectory points CSV to:", output_csv)
print(trajectory_df.head())